<a href="https://colab.research.google.com/github/sjagadee/py_torch_from_basics/blob/main/02_autograd_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import torch
# Verify torch is available
print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.11.0+cu128


# Understanding Automatic Differentiation with PyTorch

This notebook demonstrates how to calculate derivatives (gradients) both manually and using PyTorch's automatic differentiation (`autograd`) system. We'll cover simple expressions, complex functions, a single perceptron, and important concepts like gradient accumulation and disabling gradients.

## Calculate differentiation - manully

- Straight forward for simple expression `y = x^2`

## 1. Simple Expression: `y = x^2`

First, let's calculate the derivative manually. The derivative of `y = x^2` with respect to `x` is `dy/dx = 2x`.

In [28]:
def dy_dx(x):
    return 2 * x

In [29]:
dy_dx(3)

6

### Using PyTorch for `y = x^2`

PyTorch's `autograd` system can automatically calculate these derivatives. We need to define `x` as a `torch.tensor` and set `requires_grad=True` to tell PyTorch to track operations on this tensor for gradient calculation.

### Using PyTorch



In [30]:
x = torch.tensor(3.0, requires_grad=True)
# `requires_grad=True` tells PyTorch to track all operations on this tensor for backpropagation.
print(f"Input x: {x}")

Input x: 3.0


In [31]:
y = x ** 2
# Define the operation. PyTorch builds a computational graph here.
print(f"Output y: {y}")

Output y: 9.0


In [32]:
x

tensor(3., requires_grad=True)

In [33]:
y

tensor(9., grad_fn=<PowBackward0>)

In [34]:
# To fix the RuntimeError, ensure we only call .backward() once per graph unless retain_graph=True is used.
y.backward()
print("Backpropagation complete.")

Backpropagation complete.


In [35]:
# The computed gradient dy/dx (which is 2*x = 2*3 = 6) is stored in the .grad attribute.
print(f"Gradient of x (dy/dx): {x.grad}")

Gradient of x (dy/dx): 6.0


## For complex equartions

`y = x^2` and `z = sin(y)`

## 2. Complex Equation: `y = x^2`, `z = sin(y)`

Now, let's consider a more complex function where we need to apply the chain rule manually. We want to find `dz/dx`.

Given `y = x^2` and `z = sin(y)`:
- `dy/dx = 2x`
- `dz/dy = cos(y)`

By the chain rule, `dz/dx = dz/dy * dy/dx = cos(y) * 2x`.
Substituting `y = x^2`, we get `dz/dx = cos(x^2) * 2x`.

In [36]:
import math

def dz_dx(x):
    # Manual calculation of dz/dx using the chain rule
    # dz/dx = (dz/dy) * (dy/dx) = cos(y) * 2x
    # where y = x**2
    return 2 * x * (math.cos(x**2))

In [37]:
dz_dx(6)

-1.535564275528856

### Using PyTorch for `z = sin(y)`

PyTorch handles the chain rule automatically. We just need to define the operations.

### Using PyTorch

In [38]:
x = torch.tensor(6.0, requires_grad=True)

In [39]:
x = torch.tensor(6.0, requires_grad=True)
print(f"Initialized x for chain rule: {x}")

Initialized x for chain rule: 6.0


In [40]:
y = x ** 2
z = torch.sin(y)

In [41]:
y = x ** 2
z = torch.sin(y)
print(f"y = x^2 = {y.item()}")
print(f"z = sin(y) = {z.item()}")

y = x^2 = 36.0
z = sin(y) = -0.9917788505554199


In [42]:
x

tensor(6., requires_grad=True)

In [43]:
y

tensor(36., grad_fn=<PowBackward0>)

In [44]:
z

tensor(-0.9918, grad_fn=<SinBackward0>)

In [45]:
z.backward()
print("Backpropagation for z complete.")

Backpropagation for z complete.


In [46]:
x.grad

tensor(-1.5356)

In [47]:
print(f"dz/dx at x=6: {x.grad}")
# Verification: 2 * 6 * cos(36) ≈ 12 * -0.1279 ≈ -1.535

dz/dx at x=6: -1.5355643033981323


## For a Single Perceptron

## 3. Gradients for a Single Perceptron (Binary Cross-Entropy Loss)

Let's apply differentiation concepts to a simple neural network component: a single perceptron with a sigmoid activation function and binary cross-entropy loss. We'll manually calculate the gradients for the weight (`w`) and bias (`b`).

**Perceptron Model:**
- `z = w * x + b`
- `y_pred = sigmoid(z)`

**Binary Cross-Entropy Loss:**
- `L = -(y_true * log(y_pred) + (1 - y_true) * log(1 - y_pred))`

In [48]:
# inputs
x = torch.tensor(6.7) # Input feature
y = torch.tensor(0.0) # True Label (binary: 0 or 1)

In [49]:
import torch
# Parameters we want to optimize
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
print(f"Initial weight w: {w.item()}, bias b: {b.item()}")

Initial weight w: 1.0, bias b: 0.0


In [50]:
# binary cross entropy for scalar
def binary_cross_entropy(y_pred, y_true):
    epsilone = 1e-7 # A small value to prevent log(0), ensuring numerical stability
    y_pred = torch.clamp(y_pred, epsilone, 1 - epsilone) # Clamping y_pred to avoid log(0) or log(1) issues
    return -(y_true * torch.log(y_pred) + (1 - y_true) * torch.log(1 - y_pred))

In [51]:
# forward pass
z = w * x + b
y_pred = torch.sigmoid(z)

## calculate loss
loss = binary_cross_entropy(y_pred, y)

Now, let's calculate the derivatives manually using the chain rule:

- `dL/dy_pred = (y_pred - y_true) / (y_pred * (1 - y_pred))`
- `dy_pred/dz = y_pred * (1 - y_pred)` (derivative of sigmoid function)
- `dz/dw = x`
- `dz/db = 1`

Combining these for `dL/dw` and `dL/db`:
- `dL/dw = dL/dy_pred * dy_pred/dz * dz/dw = ((y_pred - y_true) / (y_pred * (1 - y_pred))) * (y_pred * (1 - y_pred)) * x = (y_pred - y_true) * x`
- `dL/db = dL/dy_pred * dy_pred/dz * dz/db = ((y_pred - y_true) / (y_pred * (1 - y_pred))) * (y_pred * (1 - y_pred)) * 1 = (y_pred - y_true) * 1`

In [52]:
# Ensure forward pass variables are available
z = w * x + b
y_pred = torch.sigmoid(z)

# Manual gradient calculations
dloss_dw = (y_pred - y) * x
dloss_db = (y_pred - y) * 1

print("Manual gradient calculation:")
print(f"dL/dw: {dloss_dw.item()}")
print(f"dL/db: {dloss_db.item()}")

Manual gradient calculation:
dL/dw: 6.691762447357178
dL/db: 0.998770534992218


In [53]:
# derivative calculations - these are the analytical derivatives based on the chain rule
# 1. dL/d(y_pred): Loss with respect to prediction
dloss_dy_pred = (y_pred - y) / (y_pred * (1 - y_pred))

# 2. d(y_pred)/dz: prediction with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. d(z)/dw: z with respect to w
dz_dw = x

# 4. d(z)/db: z with respect to b
dz_db = 1

In [54]:
dloss_dw = dloss_dy_pred * dy_pred_dz * dz_dw # Final gradient of loss with respect to weight
dloss_db = dloss_dy_pred * dy_pred_dz * dz_db # Final gradient of loss with respect to bias

In [55]:
print("Manual gradient calculation:")
print("dL/dw:", dloss_dw.item())
print("dL/db:", dloss_db.item())

Manual gradient calculation:
dL/dw: 6.691762447357178
dL/db: 0.998770534992218


### Using PyTorch for a Single Perceptron

Now let's use PyTorch's `autograd` to verify these manual calculations. We need to ensure `w` and `b` have `requires_grad=True`.

## Using PyTorch

In [56]:
import torch
# Resetting for PyTorch Autograd verification
x = torch.tensor(6.7)
y = torch.tensor(0.0)
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

# Forward pass
z = w * x + b
y_pred = torch.sigmoid(z)
loss = binary_cross_entropy(y_pred, y)

# Backward pass
loss.backward()

print("PyTorch gradients:")
print(f"w.grad: {w.grad.item()}")
print(f"b.grad: {b.grad.item()}")

PyTorch gradients:
w.grad: 6.6917619705200195
b.grad: 0.9987704753875732


In [57]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)

In [58]:
w = torch.tensor(1.0, requires_grad=True) # Set requires_grad=True for parameters we want to optimize
b = torch.tensor(0.0, requires_grad=True) # Set requires_grad=True for parameters we want to optimize

In [59]:
print(w, b)

tensor(1., requires_grad=True) tensor(0., requires_grad=True)


In [60]:
z = w*x + b
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [61]:
y_pred = torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [62]:
loss = binary_cross_entropy(y_pred, y) # Calculate the loss using the defined function

In [63]:
loss.backward() # Compute gradients of the loss with respect to all tensors that have requires_grad=True

In [64]:
w.grad

tensor(6.6918)

In [65]:
b.grad

tensor(0.9988)

## 4. Vector Input Tensors

PyTorch's `autograd` works seamlessly with vector and matrix operations. Here we compute the gradient of the mean of `x^2` with respect to `x`.

If `y = (1/n) * sum(x_i^2)`, then `dy/dx_i = (1/n) * 2 * x_i`.

# Vector input tensors

In [66]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True) # A tensor with multiple elements

In [67]:
y = (x ** 2).mean() # Calculate the mean of the squared elements

In [68]:
y.backward() # Compute gradients

In [69]:
x.grad # Each element of x.grad will be dy/dx_i

tensor([0.6667, 1.3333, 2.0000])

# Clearing Gradients

## 5. Clearing Gradients

In training loops, gradients accumulate by default. This means if you call `.backward()` multiple times on the same tensor without clearing the gradients, they will sum up. This is usually not desired, as you want the gradient for the current batch/step only. You must explicitly clear them using `zero_()`.

In [70]:
x = torch.tensor(2.0, requires_grad=True) # Initialize x

Running next 3 cells multiple times can simulate gradient accumulation

- During traning Neural network, we go through forward and backward pass multiple times
- It is not desireable to accumulate these gradients accross multiple runs
- Instead of updating, it gets accumulated.

If you run the next three cells (defining `y`, calling `y.backward()`, and checking `x.grad`) multiple times, you'll notice `x.grad` continues to increase. This is because gradients are *accumulated* by default.

In [71]:
y = x ** 3
y

tensor(8., grad_fn=<PowBackward0>)

In [72]:
y.backward()


In [73]:
x.grad

tensor(12.)

In [74]:
# To clear accumulated gradients, use .zero_()
x.grad.zero_() # Resets the gradients to zero

tensor(0.)

# Disable Gradients

## 6. Disabling Gradients

There are situations where you might want to temporarily stop tracking gradients, even for tensors that have `requires_grad=True`. This is useful during evaluation/inference, or when you're updating model parameters and don't want these updates to be part of the gradient computation graph.

Let's re-initialize `x` and see its gradient behavior initially.

In [75]:
x = torch.tensor(2.0, requires_grad=True) # x is tracking gradients

In [76]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [77]:
y.backward()

In [78]:
x.grad

tensor(4.)

Options to disable gradients:
- require_grad_(False)
- .detach()
- torch.no_grad()

PyTorch provides several ways to disable gradient tracking:

In [79]:
# 1. `x.requires_grad_(False)`: Permanently changes the `requires_grad` attribute of a tensor.
x.requires_grad_(False) # x will no longer track gradients
print(x)
y = x ** 2 # y will not have a grad_fn because x is not tracking gradients
print(y)
# y.backward() cannot be called because no computational graph was built for y
# print(x.grad) # This would be None or raise an error if y.backward() wasn't called on a graph.

tensor(2.)
tensor(4.)


In [80]:
# 2. `.detach()`: Creates a new tensor that shares the same data as the original but does NOT track gradients.
x = torch.tensor(2.0, requires_grad=True)
print(x)
z = x.detach() # z is a new tensor, detached from the computation graph
print(z)

y = x ** 2 # y still tracks gradients from x
print(y)

y1 = z ** 2 # y1 does not track gradients because z does not
print(y1)

y.backward() # This works as y is part of the graph starting from x
print(x.grad)

# y1.backward() would raise a RuntimeError because y1 is not part of a graph that requires gradients
# print(z.grad)

tensor(2., requires_grad=True)
tensor(2.)
tensor(4., grad_fn=<PowBackward0>)
tensor(4.)
tensor(4.)


In [81]:
# 3. `torch.no_grad()`: A context manager that temporarily disables gradient tracking for all operations within its block.
x = torch.tensor(2.0, requires_grad=True)
print(x)

with torch.no_grad():
    y = x ** 2 # Operations inside this block do not build a computation graph for gradient tracking
    print(y)

# y.backward() would raise a RuntimeError here because y was created within `torch.no_grad()` context
# print(x.grad)

tensor(2., requires_grad=True)
tensor(4.)


This concludes the demonstration of manual and automatic differentiation using PyTorch. Understanding these concepts is fundamental for building and training neural networks.